In [1]:
from pathlib import Path
import sys
import inspect

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from src.annual_loss_simulation import simulate_annual_losses
from src.incident_simulation import simulate_incidents

print("Annual simulator:", inspect.signature(simulate_annual_losses))
print("Incident simulator:", inspect.signature(simulate_incidents))

Annual simulator: (n_years=100000, seed=20261025, campaign_rate=3.0, success_alpha=0.36, success_beta=4.48, config=None)
Incident simulator: (n_incidents=10000, seed=42, config=None, include_details=True)


In [2]:
annual = simulate_annual_losses(
    n_years=100000,
    seed=20261025,
)

incidents = simulate_incidents(
    n_incidents=20000,
    seed=20261112,
    include_details=True,
)

print("Annual result type:", type(annual))
print("Incident result type:", type(incidents))

if isinstance(annual, dict):
    print("Annual keys:", list(annual.keys()))
elif isinstance(annual, pd.DataFrame):
    display(annual.head())

if isinstance(incidents, dict):
    print("Incident keys:", list(incidents.keys()))
elif isinstance(incidents, pd.DataFrame):
    display(incidents.head())

Annual result type: <class 'pandas.DataFrame'>
Incident result type: <class 'pandas.DataFrame'>


,year_id,campaign_count,successful_incident_count,primary_loss_usd,secondary_response_planning_loss_usd,secondary_fines_judgement_loss_usd,secondary_reputation_loss_usd,total_annual_loss_usd,maximum_single_event_loss_usd
0,1,1,0,0.000000,0.000000,0.0,0.0,0.000000,0.000000
1,2,2,0,0.000000,0.000000,0.0,0.0,0.000000,0.000000
2,3,4,1,106344.368187,92635.843005,0.0,630000.0,828980.211193,828980.211193
3,4,1,0,0.000000,0.000000,0.0,0.0,0.000000,0.000000
4,5,2,0,0.000000,0.000000,0.0,0.0,0.000000,0.000000


,incident_id,technical_severity,affected_customers,primary_loss_usd,secondary_response_planning_loss_usd,secondary_fines_judgement_loss_usd,secondary_reputation_loss_usd,total_incident_loss_usd,indicator_public_visibility,indicator_notification,...,reputation_unaffected_churn_probability,reputation_lost_deal_probability,reputation_affected_churn_count,reputation_unaffected_churn_count,reputation_total_churn_count,reputation_otherwise_won_opportunities,reputation_lost_deals_count,reputation_affected_churn_loss_usd,reputation_unaffected_churn_loss_usd,reputation_lost_deals_loss_usd
0,1,1.370097,2325,221084.072004,171612.506972,0.0,1050000,1.442697e+06,False,True,...,0.000225,0.009448,5,0,5,39,0,1050000,0,0
1,2,-1.322758,741,74917.046806,133018.457644,0.0,0,2.079355e+05,False,True,...,0.000111,0.003191,0,0,0,38,0,0,0,0
2,3,-0.133478,1682,92567.531864,169113.871362,0.0,840000,1.101681e+06,False,True,...,0.000151,0.005158,3,0,3,30,1,630000,0,210000
3,4,-1.321810,440,104406.230224,30136.599068,0.0,210000,3.445428e+05,False,False,...,0.000111,0.003192,1,0,1,39,0,210000,0,0
4,5,-1.757012,1043,117071.534884,119494.684148,0.0,0,2.365662e+05,False,True,...,0.000099,0.002677,0,0,0,40,0,0,0,0


In [3]:

# Export simulation data for the Streamlit dashboard

output_dir = PROJECT_ROOT / "outputs" / "simulations"
output_dir.mkdir(parents=True, exist_ok=True)

annual_path = output_dir / "dashboard_annual_losses.csv"
incident_path = output_dir / "dashboard_incident_losses.csv"

annual.to_csv(annual_path, index=False)
incidents.to_csv(incident_path, index=False)

# Verify accounting identities
annual_components = [
    "primary_loss_usd",
    "secondary_response_planning_loss_usd",
    "secondary_fines_judgement_loss_usd",
    "secondary_reputation_loss_usd",
]

incident_components = annual_components.copy()

assert np.allclose(
    annual[annual_components].sum(axis=1),
    annual["total_annual_loss_usd"],
)

assert np.allclose(
    incidents[incident_components].sum(axis=1),
    incidents["total_incident_loss_usd"],
)

print("Saved:", annual_path.relative_to(PROJECT_ROOT))
print("Saved:", incident_path.relative_to(PROJECT_ROOT))
print("Annual simulation rows:", len(annual))
print("Incident simulation rows:", len(incidents))
print("All accounting checks passed.")


Saved: outputs/simulations/dashboard_annual_losses.csv
Saved: outputs/simulations/dashboard_incident_losses.csv
Annual simulation rows: 100000
Incident simulation rows: 20000
All accounting checks passed.
